# Megaminx API из Jupyter

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/poleschukfa/MegaminxRobot/blob/main/notebooks/megaminx_api.ipynb)

Управление роботом **параллельно с viewer** через HTTP (`MegaminxClient`).

> **Авторизация:** для HTTP-запросов нужен Bearer-токен — скопируйте его во [viewer → вкладка API](/viewer.html), войдите через `POST /api/auth/login` или используйте серверный `API_SECRET`. Подробности в разделе **«Авторизация HTTP API»** ниже.

## Что нужно запустить

1. **Робот:** `robot/online.py` + `robot/robot_streamer.py`
2. **Сервер:** `node server.js` (roborubiks.ru)
3. **Viewer:** открыт в браузере, вы **активный зритель** (первая позиция в очереди)

## Установка

**Google Colab:** нажмите бейдж «Open In Colab» → **Runtime → Run all** (или сначала ячейку **«установка»**, затем **«авторизация»**).

**Локально:**

```bash
pip install requests certifi
```

Скопируйте `megaminx_client.py` из репозитория или добавьте путь к нему в `sys.path`.

> **macOS + SSL:** `pip install certifi` или `MegaminxClient(..., verify_ssl=False)`

## Методы клиента

| Метод | Описание |
|-------|----------|
| `health()` | Стример, очередь, activeViewer |
| `features()` | Версия API, поддержка solve_state |
| `help()` | Список команд |
| `get_available_faces()` | Грани U, D, F, … |
| `rotate(face, direction)` | Один ход (`cw` / `ccw`) |
| `get_state()` | Состояние робота (история на моторах) |
| `get_history()` | История ходов |
| `get_path()` | Путь и обратный путь по истории робота |
| `get_reverse_path()` | Только обратный путь (робот) |
| `get_solve_state()` | Состояние **симулятора** (кнопка «Состояние») |
| `execute_path(path)` | Выполнить последовательность |
| `go_to_init()` | Собрать обратно (обратные ходы) |
| `reset_history()` | Сбросить историю без вращения |
| `command(name, params)` | Любая команда (напр. `test`) |


In [ ]:
# Ячейка 0 — установка (в Colab запустите первой!)
import os
import sys
import subprocess

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "certifi"])
import requests

WORKDIR = os.getcwd()
CLIENT_FILE = os.path.join(WORKDIR, "megaminx_client.py")
CLIENT_URL = "https://roborubiks.ru/notebooks/megaminx_client.py"

def _download_client():
    r = requests.get(
        CLIENT_URL,
        timeout=60,
        headers={"User-Agent": "Mozilla/5.0 (Megaminx Colab)"},
    )
    r.raise_for_status()
    if "class MegaminxClient" not in r.text:
        raise RuntimeError(f"Неверный ответ: {CLIENT_URL}")
    with open(CLIENT_FILE, "w", encoding="utf-8") as f:
        f.write(r.text)

if not os.path.isfile(CLIENT_FILE) or os.path.getsize(CLIENT_FILE) < 1000:
    try:
        _download_client()
    except Exception:
        subprocess.check_call(["curl", "-fsSL", "-o", CLIENT_FILE, CLIENT_URL])
    if not os.path.isfile(CLIENT_FILE) or os.path.getsize(CLIENT_FILE) < 1000:
        raise RuntimeError(
            "Не удалось скачать megaminx_client.py. "
            "Скачайте вручную: " + CLIENT_URL
        )

if WORKDIR not in sys.path:
    sys.path.insert(0, WORKDIR)

from megaminx_client import MegaminxClient, MegaminxAPIError
print("OK: MegaminxClient загружен из", CLIENT_FILE)


## Авторизация HTTP API

Если на сервере включена авторизация (`GET /api/auth/config` → `"apiAuthRequired": true`), каждый запрос к `/api/*` требует заголовок:

`Authorization: Bearer <token>`

### Способ 1 — токен из viewer (удобнее)

1. Откройте [viewer](/viewer.html) и **войдите** в аккаунт.
2. Станьте **активным зрителем** (если будете слать команды роботу).
3. В пульте откройте вкладку **API** → блок **«Токен для API»** → **Копировать**.

Токен — сессия вашего логина. Действует до выхода из viewer.

### Способ 2 — логин через API

`POST /api/auth/login` с телом `{"username": "...", "password": "..."}` → в ответе поле `"token"`.

### Способ 3 — серверный API_SECRET

Для админ-скриптов: `export API_SECRET=...` (значение в `.env` на сервере).

> **Важно:** токен открывает API, но **команды робота** работают только когда вы **активный зритель** в очереди viewer.

In [ ]:
import os
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "requests", "certifi"])

import requests

BASE_URL = "https://roborubiks.ru"
# BASE_URL = "http://localhost:3000"

# A) Вставьте токен из viewer → вкладка API
API_TOKEN = os.environ.get("MEGAMINX_API_TOKEN") or "ВСТАВЬТЕ_ТОКЕН_ИЗ_VIEWER"

# B) Или логин (раскомментируйте):
# login = requests.post(
#     f"{BASE_URL}/api/auth/login",
#     json={"username": "your_login", "password": "your_password"},
#     timeout=30,
# )
# login.raise_for_status()
# API_TOKEN = login.json()["token"]

# C) Или серверный секрет: export API_SECRET=...
# API_TOKEN = os.environ.get("API_SECRET") or API_TOKEN

auth_cfg = requests.get(f"{BASE_URL}/api/auth/config", timeout=15).json()
print("auth config:", auth_cfg)
print("token set:", bool(API_TOKEN) and API_TOKEN != "ВСТАВЬТЕ_ТОКЕН_ИЗ_VIEWER")

api = MegaminxClient(BASE_URL, api_token=API_TOKEN)
# api = MegaminxClient(BASE_URL, api_token=API_TOKEN, verify_ssl=False)  # macOS без certifi


## 1. Проверка подключения

In [ ]:
health = api.health()
features = api.features()

print("health:", health)
print("features:", features)

if not health.get("broadcaster"):
    print("⚠️ robot_streamer не подключён")
elif not health.get("activeViewer"):
    print("⚠️ Нет активного зрителя — откройте viewer и встаньте в очередь")
else:
    print("✅ Готов к командам")

if not features.get("features", {}).get("solve_state"):
    print("⚠️ get_solve_state недоступен — перезапустите node server.js")

## 2. Справка (help)

In [ ]:
help_info = api.help()
print("Нотация:", help_info.get("notation", ""), "\n")
for name, desc in help_info.get("commands", {}).items():
    print(f"{name:22} {desc}")

## 3. Доступные грани

In [ ]:
faces = api.get_available_faces()
print("Грани:", faces.get("faces"))

## 4. Один ход — `rotate`

Грани: `U, D, F, B, L, R, BL, BR, FL, FR, DL, DR`. Направление: `cw` или `ccw`.

In [ ]:
try:
    r = api.rotate("U", "cw")
    print("Ход:", r.get("move", {}).get("notation"))
except MegaminxAPIError as e:
    print("Ошибка:", e, e.payload)

## 5. Состояние робота

История ходов на **физическом** роботе (не симулятор).

In [ ]:
state = api.get_state()
print("history_length:", state.get("state", {}).get("history_length"))
print("last_move:", state.get("state", {}).get("last_move"))

history = api.get_history()
notations = [m.get("notation") for m in history.get("history", [])]
print(f"История ({len(notations)}):", notations[-10:])

## 6. Путь робота — `get_path` / `get_reverse_path`

In [ ]:
paths = api.get_path()
print("path:", paths.get("path"))
print("reverse_path:", paths.get("reverse_path"))

rev = api.get_reverse_path()
print("get_reverse_path:", rev.get("reverse_path"))

## 7. Состояние симулятора — `get_solve_state`

Как кнопка **«↩️ Состояние»** в viewer (solver по цветам в iframe).

- `solve_path` — как собрать из текущего состояния
- `reverse_path` — обратная последовательность (в поле пути viewer)

In [ ]:
try:
    sim = api.get_solve_state()
    if sim.get("solved"):
        print("✅", sim.get("message", "уже собрано"))
    else:
        print("solve_path:  ", sim.get("solve_path"))
        print("reverse_path:", sim.get("reverse_path"))
except MegaminxAPIError as e:
    print("Ошибка:", e)

## 8. Выполнить путь — `execute_path`

Viewer показывает каждый ход в симуляторе. Ответ содержит `time` (секунды) и `executed`.

In [ ]:
PATH = "U.F.R'"  # измените

try:
    r = api.execute_path(PATH)
    print(f"✅ {r.get('executed')} ходов за {r.get('time')} с")
except MegaminxAPIError as e:
    print("Ошибка:", e, e.payload)

## 9. Собрать обратно — `go_to_init`

Выполняет обратные ходы по истории робота и сбрасывает историю.

In [ ]:
# Раскомментируйте для запуска:
# r = api.go_to_init()
# print(r)

## 10. Сброс истории — `reset_history`

Только обнуляет историю в памяти, **без** вращения моторов.

In [ ]:
# r = api.reset_history()
# print(r)

## 11. Произвольная команда — `command`

Например, тест моторов (если поддерживается роботом):

In [ ]:
# r = api.command("test")
# print(r)

## 12. HTTP без обёртки (requests)

Заголовок авторизации (если `apiAuthRequired`):

```python
headers = {"Authorization": f"Bearer {API_TOKEN}"}
```

| Метод | URL |
|-------|-----|
| GET | `/health` (без токена) |
| GET | `/api/auth/config` (без токена) |
| POST | `/api/auth/login` — `{"username","password"}` → `token` |
| GET | `/api/features` |
| GET | `/api/state` |
| GET | `/api/history` |
| GET | `/api/solve-state` |
| POST | `/api/command` — `{ "command": "...", "params": {} }` |

In [ ]:
import requests
import certifi

BASE = BASE_URL
verify = certifi.where()
headers = {"Authorization": f"Bearer {API_TOKEN}"}

# requests.get(f"{BASE}/health", verify=verify).json()
# requests.get(f"{BASE}/api/solve-state", headers=headers, verify=verify, timeout=120).json()
# requests.post(f"{BASE}/api/command", headers=headers, json={
#     "command": "rotate",
#     "params": {"face": "F", "direction": "ccw"},
# }, verify=verify, timeout=30).json()